In [0]:
# DBTITLE 1, Rebuild voc_llm_topic_classification_preferred_all
# ============================================================
# voc_llm_topic_classification 전체 기준 preferred_all 재생성
# - 기존 memo_id 보정 로직과 동일한 방식
# - r10, r11 포함 전체 llm_round 대상
# - memo_id별 preferred 1건 선정
# - topic 누락 시 fallback topic 보정
# - 결과 테이블 overwrite
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window

SAVE_DB = "sandbox.z_jungryo_lee"

CLASSIFIED_TABLE = "sandbox.t_online_voc_analysis.voc_llm_topic_classification"
CLASSIFIED_PREFERRED_TABLE = f"{SAVE_DB}.voc_llm_topic_classification_preferred_all"

print("[CONFIG]")
print(f"  CLASSIFIED_TABLE = {CLASSIFIED_TABLE}")
print(f"  CLASSIFIED_PREFERRED_TABLE = {CLASSIFIED_PREFERRED_TABLE}")


def normalize_memo_expr(col_name: str):
    return F.trim(
        F.regexp_replace(
            F.translate(F.col(col_name).cast("string"), "　", " "),
            "\\s+",
            " ",
        )
    )


def with_memo_id(df):
    return df.withColumn(
        "memo_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("cate_1_depth").cast("string"), F.lit("")),
                F.coalesce(F.col("cate_2_depth").cast("string"), F.lit("")),
                F.coalesce(F.col("sc_measurement").cast("string"), F.lit("")),
                normalize_memo_expr("memo"),
            ),
            256,
        ),
    )


def extract_round_num_expr(col_name: str):
    return F.when(
        F.regexp_extract(F.lower(F.col(col_name)), r"(\d+)", 1) != "",
        F.regexp_extract(F.lower(F.col(col_name)), r"(\d+)", 1).cast("int"),
    )


if not spark.catalog.tableExists(CLASSIFIED_TABLE):
    raise RuntimeError(f"[ERROR] 원본 테이블 없음: {CLASSIFIED_TABLE}")


# ------------------------------------------------------------
# 1. classified table 로드 + memo_id 보정
# ------------------------------------------------------------
classified_df = (
    spark.table(CLASSIFIED_TABLE)
    .where(F.col("memo").isNotNull())
)

classified_repaired_df = (
    classified_df
    .withColumn("memo_norm", normalize_memo_expr("memo"))
    .transform(with_memo_id)
    .withColumn("llm_round_num", extract_round_num_expr("llm_round"))
)

print(f"[LOAD] classified rows: {classified_repaired_df.count():,}")


# ------------------------------------------------------------
# 2. memo_id 중복 현황 확인
# ------------------------------------------------------------
overlap_stats_df = (
    classified_repaired_df
    .groupBy("memo_id")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("llm_round").alias("llm_round_count"),
    )
    .where(F.col("row_count") > 1)
)

print(f"[OVERLAP] 중복 memo_id 수: {overlap_stats_df.count():,}")

display(
    overlap_stats_df
    .orderBy(F.desc("row_count"), F.desc("llm_round_count"))
    .limit(20)
)


# ------------------------------------------------------------
# 3. preferred 선정 우선순위
# 기존 memo_id 보정 파일 로직 유지:
# - r2, r1, r0 우선
# - 그 외 round는 최신 round_num 우선
# - data_created_dt 최신 우선
# ------------------------------------------------------------
preferred_window = Window.partitionBy("memo_id").orderBy(
    F.when(F.col("llm_round_num") == 2, F.lit(0))
     .when(F.col("llm_round_num") == 1, F.lit(1))
     .when(F.col("llm_round_num") == 0, F.lit(2))
     .otherwise(F.lit(3)).asc(),
    F.when(F.col("llm_round_num").isin(2, 1, 0), F.col("llm_round_num")).desc_nulls_last(),
    F.when(~F.col("llm_round_num").isin(2, 1, 0), F.col("llm_round_num")).desc_nulls_last(),
    F.col("data_created_dt").desc_nulls_last(),
    F.col("llm_round").desc_nulls_last(),
)


classified_preferred_df = (
    classified_repaired_df
    .withColumn("memo_pick_rank", F.row_number().over(preferred_window))
    .where(F.col("memo_pick_rank") == 1)
    .drop("memo_pick_rank", "llm_round_num", "memo_norm")
)

print(f"[PREFERRED] selected rows: {classified_preferred_df.count():,}")


# ------------------------------------------------------------
# 4. topic fallback
# preferred row의 topic이 비어 있으면 같은 memo_id 내 유효 topic으로 보정
# ------------------------------------------------------------
fallback_topic_df = (
    classified_repaired_df
    .where(F.col("topic").isNotNull() & (F.length(F.trim(F.col("topic"))) > 0))
    .withColumn("topic_fallback_rank", F.row_number().over(preferred_window))
    .where(F.col("topic_fallback_rank") == 1)
    .select(
        "memo_id",
        F.col("topic").alias("fallback_topic"),
    )
)

topic_missing_condition = (
    F.col("p.topic").isNull()
    | (F.length(F.trim(F.col("p.topic"))) == 0)
)

classified_preferred_filled_df = (
    classified_preferred_df.alias("p")
    .join(fallback_topic_df.alias("f"), on="memo_id", how="left")
    .withColumn(
        "topic",
        F.when(topic_missing_condition, F.col("f.fallback_topic"))
         .otherwise(F.col("p.topic"))
    )
    .drop("fallback_topic")
)

filled_topic_count = (
    classified_preferred_df.alias("p")
    .join(fallback_topic_df.alias("f"), on="memo_id", how="inner")
    .where(topic_missing_condition & F.col("f.fallback_topic").isNotNull())
    .count()
)

print(f"[FALLBACK] topic 보정 건수: {filled_topic_count:,}")


# ------------------------------------------------------------
# 5. preferred_all overwrite 저장
# ------------------------------------------------------------
classified_preferred_filled_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(CLASSIFIED_PREFERRED_TABLE)

print(f"[SAVE] {CLASSIFIED_PREFERRED_TABLE}: {classified_preferred_filled_df.count():,}건")


# ------------------------------------------------------------
# 6. 검증
# ------------------------------------------------------------
print("\n[VERIFY] preferred_all by llm_round")

display(
    spark.table(CLASSIFIED_PREFERRED_TABLE)
    .groupBy("llm_round")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("memo_id").alias("memo_id_count"),
        F.sum(F.when(F.col("memo_id").isNull(), 1).otherwise(0)).alias("memo_id_null_count"),
        F.sum(F.when(F.col("topic").isNull() | (F.length(F.trim(F.col("topic"))) == 0), 1).otherwise(0)).alias("topic_null_count"),
        F.sum(F.when(F.col("topic_rev").isNull(), 1).otherwise(0)).alias("topic_rev_null_count"),
    )
    .orderBy("llm_round")
)

print("\n[VERIFY] r10/r11 only")

display(
    spark.table(CLASSIFIED_PREFERRED_TABLE)
    .where(F.col("llm_round").isin("r10", "r11"))
    .groupBy("llm_round")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("memo_id").alias("memo_id_count"),
        F.sum(F.when(F.col("memo_id").isNull(), 1).otherwise(0)).alias("memo_id_null_count"),
        F.sum(F.when(F.col("topic_rev").isNull(), 1).otherwise(0)).alias("topic_rev_null_count"),
    )
    .orderBy("llm_round")
)

print("\n✅ voc_llm_topic_classification_preferred_all 전체 재생성 완료")